# Bug Prediction — Comparison Experiment

**Research Question:** Does bug prediction generalize across languages and time periods?

## 3 Comparisons:
1. **Cross-Language** — Python model vs TypeScript model
2. **Cross-Time** — Old era (2018-2020) vs New era (2024-2026)
3. **Universal vs Specific** — Combined model vs language-specific

**Metric:** AUC drop — how much accuracy is lost when testing on unseen data

---

## Step 1 — Install & Import

In [ ]:
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib shap -q
print('✅ Done')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from xgboost import XGBClassifier
print('✅ Imports done')

## Step 2 — Load Data

In [ ]:
df_py  = pd.read_csv('combined-py-training.csv')
df_ts  = pd.read_csv('combined-ts-training.csv')
df_all = pd.concat([df_py, df_ts], ignore_index=True)

# Encode time_period
le = LabelEncoder()
df_py  = df_py.copy();  df_py['time_enc']  = le.fit_transform(df_py['time_period'])
df_ts  = df_ts.copy();  df_ts['time_enc']  = le.fit_transform(df_ts['time_period'])
df_all = df_all.copy(); df_all['time_enc'] = le.fit_transform(df_all['time_period'])
df_all['lang_enc'] = (df_all['language_group'] == 'TypeScript').astype(int)

print('=== DATA LOADED ===')
print(f'Python CSV      : {len(df_py):,} rows | {df_py["is_buggy"].sum()} bugs ({df_py["is_buggy"].mean():.1%})')
print(f'TypeScript CSV  : {len(df_ts):,} rows | {df_ts["is_buggy"].sum()} bugs ({df_ts["is_buggy"].mean():.1%})')
print(f'Combined        : {len(df_all):,} rows | {df_all["is_buggy"].sum()} bugs ({df_all["is_buggy"].mean():.1%})')
print(f'Time periods    : {sorted(df_all["time_period"].unique())}')

## Step 3 — Feature Setup & Helper Functions

In [ ]:
FEATURES_LANG = [
    'prior_bugs_author', 'avg_complexity', 'test_ratio',
    'test_files_changed', 'complexity_per_file',
    'files_changed', 'num_methods', 'churn_ratio',
    'lines_added', 'lines_deleted',
    'commit_hour', 'day_of_week', 'is_weekend', 'is_night_commit',
    'time_enc',
]
FEATURES_UNIV = FEATURES_LANG + ['lang_enc']
TARGET = 'is_buggy'
PERIODS = ['2018-2020', '2021-2023', '2024-2026']

def make_model(X_train, y_train):
    scale = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rf = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            max_depth=10, min_samples_leaf=5,
            random_state=42, n_jobs=-1))
    ])
    xgb = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('clf', XGBClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.05,
            scale_pos_weight=scale, random_state=42,
            eval_metric='auc', verbosity=0))
    ])
    rf_auc  = cross_val_score(rf,  X_train, y_train, cv=cv, scoring='roc_auc').mean()
    xgb_auc = cross_val_score(xgb, X_train, y_train, cv=cv, scoring='roc_auc').mean()
    best = rf if rf_auc >= xgb_auc else xgb
    name = 'RandomForest' if rf_auc >= xgb_auc else 'XGBoost'
    best.fit(X_train, y_train)
    return best, name, max(rf_auc, xgb_auc)

def evaluate(model, X_test, y_test):
    if y_test.sum() == 0:
        return None
    proba = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    fpr, tpr, _ = roc_curve(y_test, proba)
    return {
        'auc':       round(roc_auc_score(y_test, proba), 4),
        'precision': round(precision_score(y_test, preds, zero_division=0), 4),
        'recall':    round(recall_score(y_test, preds, zero_division=0), 4),
        'f1':        round(f1_score(y_test, preds, zero_division=0), 4),
        'fpr': fpr, 'tpr': tpr, 'proba': proba, 'y_true': y_test,
    }

def verdict(drop):
    if drop <= 0.05:  return 'Generalizes well   (< 5% drop)'
    if drop <= 0.15:  return 'Moderate drift     (5-15% drop)'
    return                   'Poor generalization (> 15% drop)'

print(f'Features (lang-specific): {len(FEATURES_LANG)}')
print(f'Features (universal)    : {len(FEATURES_UNIV)}')

## Step 4 — Comparison 1: Cross-Language

In [ ]:
print('=' * 55)
print('  COMPARISON 1 — CROSS LANGUAGE')
print('=' * 55)
results_lang = {}

# Baseline: Python → Python
print('\n[1/4] Python → Python')
X_py = df_py[FEATURES_LANG].fillna(0); y_py = df_py[TARGET]
X_ptr, X_pte, y_ptr, y_pte = train_test_split(X_py, y_py, test_size=0.2, stratify=y_py, random_state=42)
model_py, name_py, cv_py = make_model(X_ptr, y_ptr)
r = evaluate(model_py, X_pte, y_pte)
results_lang['Python → Python'] = r
print(f'  Model: {name_py}  CV AUC: {cv_py:.3f}  Test AUC: {r["auc"]:.3f}')

# Baseline: TypeScript → TypeScript
print('\n[2/4] TypeScript → TypeScript')
X_ts = df_ts[FEATURES_LANG].fillna(0); y_ts = df_ts[TARGET]
X_ttr, X_tte, y_ttr, y_tte = train_test_split(X_ts, y_ts, test_size=0.2, stratify=y_ts, random_state=42)
model_ts, name_ts, cv_ts = make_model(X_ttr, y_ttr)
r = evaluate(model_ts, X_tte, y_tte)
results_lang['TypeScript → TypeScript'] = r
print(f'  Model: {name_ts}  CV AUC: {cv_ts:.3f}  Test AUC: {r["auc"]:.3f}')

# Cross: Python → TypeScript
print('\n[3/4] Python → TypeScript (cross)')
r = evaluate(model_py, X_ts.fillna(0), y_ts)
results_lang['Python → TypeScript'] = r
print(f'  Test AUC: {r["auc"]:.3f}')

# Cross: TypeScript → Python
print('\n[4/4] TypeScript → Python (cross)')
r = evaluate(model_ts, X_py.fillna(0), y_py)
results_lang['TypeScript → Python'] = r
print(f'  Test AUC: {r["auc"]:.3f}')

# AUC Drop
drop_py_ts = results_lang['Python → Python']['auc'] - results_lang['Python → TypeScript']['auc']
drop_ts_py = results_lang['TypeScript → TypeScript']['auc'] - results_lang['TypeScript → Python']['auc']
print(f'\n--- AUC DROP ---')
print(f'  Python→TS   : {drop_py_ts:+.3f}  {verdict(drop_py_ts)}')
print(f'  TS→Python   : {drop_ts_py:+.3f}  {verdict(drop_ts_py)}')

## Step 5 — Comparison 2: Cross-Time

In [ ]:
print('=' * 55)
print('  COMPARISON 2 — CROSS TIME (Concept Drift)')
print('=' * 55)
results_time = {}

for train_period in PERIODS:
    for test_period in PERIODS:
        label    = f'{train_period} -> {test_period}'
        train_df = df_all[df_all['time_period'] == train_period]
        test_df  = df_all[df_all['time_period'] == test_period]
        if train_df['is_buggy'].sum() < 5 or test_df['is_buggy'].sum() < 5:
            continue
        X_tr = train_df[FEATURES_LANG].fillna(0); y_tr = train_df[TARGET]
        X_te = test_df[FEATURES_LANG].fillna(0);  y_te = test_df[TARGET]
        m, _, _ = make_model(X_tr, y_tr)
        r = evaluate(m, X_te, y_te)
        if r:
            results_time[label] = r
            marker = '[SAME]' if train_period == test_period else '[CROSS]'
            print(f'  {marker} {label:<35} AUC: {r["auc"]:.3f}')

same_era_aucs = [results_time.get(f'{p} -> {p}', {}).get('auc') for p in PERIODS]
same_era_aucs = [a for a in same_era_aucs if a]
avg_same      = np.mean(same_era_aucs)
old_to_new    = results_time.get('2018-2020 -> 2024-2026', {}).get('auc')
new_to_old    = results_time.get('2024-2026 -> 2018-2020', {}).get('auc')

print(f'\n--- KEY FINDINGS ---')
print(f'  Same-era avg AUC : {avg_same:.3f}')
if old_to_new:
    d = avg_same - old_to_new
    print(f'  Old->New AUC     : {old_to_new:.3f}  drop: {d:+.3f}  {verdict(d)}')
if new_to_old:
    d = avg_same - new_to_old
    print(f'  New->Old AUC     : {new_to_old:.3f}  drop: {d:+.3f}  {verdict(d)}')

## Step 6 — Comparison 3: Universal vs Specific

In [ ]:
print('=' * 55)
print('  COMPARISON 3 — UNIVERSAL vs SPECIFIC')
print('=' * 55)
results_univ = {}

# Train universal
print('\n[1/3] Training universal model...')
X_all = df_all[FEATURES_UNIV].fillna(0); y_all = df_all[TARGET]
X_atr, X_ate, y_atr, y_ate = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)
model_univ, name_univ, cv_univ = make_model(X_atr, y_atr)
r = evaluate(model_univ, X_ate, y_ate)
results_univ['Universal -> Combined'] = r
print(f'  Model: {name_univ}  CV AUC: {cv_univ:.3f}  Test AUC: {r["auc"]:.3f}')
joblib.dump(model_univ, 'universal_bug_model.pkl')
print('  Saved: universal_bug_model.pkl')

# Universal on Python
print('\n[2/3] Universal tested on Python...')
X_py_u = df_py[FEATURES_LANG].fillna(0).copy()
X_py_u['lang_enc'] = 0
r = evaluate(model_univ, X_py_u[FEATURES_UNIV], df_py[TARGET])
results_univ['Universal -> Python'] = r
print(f'  AUC: {r["auc"]:.3f}')

# Universal on TypeScript
print('\n[3/3] Universal tested on TypeScript...')
X_ts_u = df_ts[FEATURES_LANG].fillna(0).copy()
X_ts_u['lang_enc'] = 1
r = evaluate(model_univ, X_ts_u[FEATURES_UNIV], df_ts[TARGET])
results_univ['Universal -> TypeScript'] = r
print(f'  AUC: {r["auc"]:.3f}')

spec_py_auc = results_lang['Python -> Python']['auc'] if 'Python -> Python' in results_lang else results_lang['Python → Python']['auc']
spec_ts_auc = results_lang['TypeScript -> TypeScript']['auc'] if 'TypeScript -> TypeScript' in results_lang else results_lang['TypeScript → TypeScript']['auc']
univ_py_auc = results_univ['Universal -> Python']['auc']
univ_ts_auc = results_univ['Universal -> TypeScript']['auc']

print(f'\n--- UNIVERSAL vs SPECIFIC ---')
print(f'  Python     Specific: {spec_py_auc:.3f}  Universal: {univ_py_auc:.3f}  Diff: {univ_py_auc-spec_py_auc:+.3f}')
print(f'  TypeScript Specific: {spec_ts_auc:.3f}  Universal: {univ_ts_auc:.3f}  Diff: {univ_ts_auc-spec_ts_auc:+.3f}')

## Step 7 — Comparison Chart

In [ ]:
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('Bug Prediction — Full Comparison Report', fontsize=16, fontweight='bold')

C = {'py':'#3498db','ts':'#e67e22','univ':'#2ecc71','cross':'#e74c3c'}

# Plot 1: Cross-Language AUC
ax1 = fig.add_subplot(gs[0, 0])
lang_labels = list(results_lang.keys())
lang_aucs   = [results_lang[k]['auc'] for k in lang_labels]
colors1 = [C['py'], C['ts'], C['cross'], C['cross']]
bars = ax1.bar(range(len(lang_labels)), lang_aucs, color=colors1, edgecolor='white', linewidth=1.5)
ax1.set_xticks(range(len(lang_labels)))
ax1.set_xticklabels(lang_labels, rotation=20, ha='right', fontsize=8)
ax1.set_title('Comparison 1\nCross-Language AUC', fontweight='bold')
ax1.set_ylabel('AUC'); ax1.set_ylim(0.4, 1.0)
ax1.axhline(0.5, color='red', linestyle='--', alpha=0.4)
for bar, val in zip(bars, lang_aucs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

# Plot 2: AUC Drop
ax2 = fig.add_subplot(gs[0, 1])
drop_vals   = [drop_py_ts, drop_ts_py]
drop_labels = ['Python→TS', 'TS→Python']
drop_colors = ['#e74c3c' if d > 0.15 else '#e67e22' if d > 0.05 else '#2ecc71' for d in drop_vals]
bars2 = ax2.bar(drop_labels, drop_vals, color=drop_colors, edgecolor='white', linewidth=1.5)
ax2.axhline(0.05, color='orange', linestyle='--', alpha=0.7, label='5% threshold')
ax2.axhline(0.15, color='red',    linestyle='--', alpha=0.7, label='15% threshold')
ax2.set_title('Comparison 1\nAUC Drop', fontweight='bold')
ax2.set_ylabel('AUC Drop'); ax2.legend(fontsize=8)
for bar, val in zip(bars2, drop_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
             f'{val:+.3f}', ha='center', fontsize=10, fontweight='bold')

# Plot 3: Cross-Time heatmap
ax3 = fig.add_subplot(gs[0, 2])
matrix = np.zeros((3, 3))
for i, tp_train in enumerate(PERIODS):
    for j, tp_test in enumerate(PERIODS):
        key = f'{tp_train} -> {tp_test}'
        matrix[i, j] = results_time.get(key, {}).get('auc', 0)
sns.heatmap(matrix, annot=True, fmt='.3f', cmap='RdYlGn',
            xticklabels=[p[:4] for p in PERIODS],
            yticklabels=[p[:4] for p in PERIODS],
            ax=ax3, vmin=0.5, vmax=1.0, linewidths=0.5)
ax3.set_title('Comparison 2\nCross-Time Heatmap\n(row=train, col=test)', fontweight='bold')
ax3.set_xlabel('Test Period'); ax3.set_ylabel('Train Period')

# Plot 4: Universal vs Specific
ax4 = fig.add_subplot(gs[1, 0])
comp_labels = ['Py\nSpecific','Py\nUniversal','TS\nSpecific','TS\nUniversal']
comp_aucs   = [spec_py_auc, univ_py_auc, spec_ts_auc, univ_ts_auc]
comp_colors = [C['py'], C['univ'], C['ts'], C['univ']]
bars4 = ax4.bar(comp_labels, comp_aucs, color=comp_colors, edgecolor='white', linewidth=1.5)
ax4.set_title('Comparison 3\nUniversal vs Specific', fontweight='bold')
ax4.set_ylabel('AUC'); ax4.set_ylim(0.4, 1.0)
for bar, val in zip(bars4, comp_aucs):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

# Plot 5: ROC Curves
ax5 = fig.add_subplot(gs[1, 1])
roc_items = [
    ('Python->Python',    results_lang['Python → Python'],     C['py'],    '-'),
    ('TS->TypeScript',    results_lang['TypeScript → TypeScript'], C['ts'],'-'),
    ('Python->TS(cross)', results_lang['Python → TypeScript'], C['cross'], '--'),
    ('TS->Py(cross)',     results_lang['TypeScript → Python'], C['cross'], ':'),
    ('Universal',         results_univ['Universal -> Combined'],    C['univ'],  '-'),
]
for label, r, color, ls in roc_items:
    if r and 'fpr' in r:
        ax5.plot(r['fpr'], r['tpr'], color=color, linewidth=2,
                 linestyle=ls, label=f'{label} ({r["auc"]:.2f})')
ax5.plot([0,1],[0,1],'k--', alpha=0.3)
ax5.set_title('ROC Curves — All Experiments', fontweight='bold')
ax5.set_xlabel('False Positive Rate'); ax5.set_ylabel('True Positive Rate')
ax5.legend(fontsize=7, loc='lower right')

# Plot 6: Summary text
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
finding1 = max(drop_py_ts, drop_ts_py)
finding2 = avg_same - (old_to_new or avg_same)
finding3 = max(univ_py_auc - spec_py_auc, univ_ts_auc - spec_ts_auc)
summary = (
    'RESEARCH FINDINGS\n' + '-'*30 + '\n\n'
    + f'1. Cross-Language\n   Drop: {finding1:+.3f}\n   {verdict(finding1)}\n\n'
    + f'2. Concept Drift\n   Drop: {finding2:+.3f}\n   {verdict(finding2)}\n\n'
    + f'3. Universal vs Specific\n   Diff: {finding3:+.3f}\n'
    + ('   Universal is BETTER\n' if finding3 >= 0 else '   Specific is BETTER\n')
    + '\n' + '-'*30 + '\n'
    + 'GUIDE\n'
    + '< 5%  drop = one model ok\n'
    + '5-15% drop = tune per lang\n'
    + '> 15% drop = separate models'
)
ax6.text(0.05, 0.95, summary, transform=ax6.transAxes,
         fontsize=9.5, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.9))

plt.savefig('comparison_report.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_report.png')

## Step 8 — Save Results & Defense Summary

In [ ]:
rows = []
for label, r in results_lang.items():
    rows.append({'Comparison':'Cross-Language','Experiment':label,
                 'AUC':r['auc'],'Precision':r['precision'],
                 'Recall':r['recall'],'F1':r['f1']})
for label, r in results_time.items():
    rows.append({'Comparison':'Cross-Time','Experiment':label,
                 'AUC':r['auc'],'Precision':r['precision'],
                 'Recall':r['recall'],'F1':r['f1']})
for label, r in results_univ.items():
    rows.append({'Comparison':'Universal','Experiment':label,
                 'AUC':r['auc'],'Precision':r['precision'],
                 'Recall':r['recall'],'F1':r['f1']})
pd.DataFrame(rows).to_csv('full_comparison_results.csv', index=False)

print('=' * 60)
print('  DEFENSE SUMMARY')
print('=' * 60)
print(f'Dataset    : {len(df_py):,} Python + {len(df_ts):,} TypeScript = {len(df_all):,} commits')
print(f'Bugs       : {df_all["is_buggy"].sum()} ({df_all["is_buggy"].mean():.1%})')
print(f'Time span  : 2018 to 2026')
print(f'\nFINDING 1 — Cross-Language:')
print(f'  Python->TS drop   : {drop_py_ts:+.3f}  {verdict(drop_py_ts)}')
print(f'  TS->Python drop   : {drop_ts_py:+.3f}  {verdict(drop_ts_py)}')
print(f'\nFINDING 2 — Concept Drift:')
print(f'  Same-era avg AUC  : {avg_same:.3f}')
if old_to_new:
    print(f'  Old->New drop     : {avg_same-old_to_new:+.3f}  {verdict(avg_same-old_to_new)}')
print(f'\nFINDING 3 — Universal vs Specific:')
print(f'  Python  diff      : {univ_py_auc-spec_py_auc:+.3f}')
print(f'  TS      diff      : {univ_ts_auc-spec_ts_auc:+.3f}')
print(f'\nSaved: full_comparison_results.csv, comparison_report.png')
print('=' * 60)